# pcefg: Point-Charge (PC) Model for the Electric Field Gradient (EFG)

`pcefg` is a Python package for computing the Electric Field Gradient (EFG) tensor, asymmetry parameters ($\eta$), and quadrupolar coupling constants ($\chi_Q$) in crystal structures using a classical point-charge model. It serves as a lightweight, fast alternative or complementary approach to First-Principles/Density Functional Theory (DFT) calculations.

---

## Features

- **Fast EFG Tensor Calculation**: Computes lattice EFG tensors via direct lattice summation.
- **Sternheimer Antishielding**: Supports polarization corrections via $(1-\gamma_\infty)$.
- **ASE Integration**: Works directly with Atomic Simulation Environment (`ase.Atoms`) structures.
- **Crystalline Symmetry Support**: Automatically handles spacegroup site labels and site-specific charge specifications.
- **Quadrupole Coupling Utilities**: Calculates $V_{zz}$, $\eta$, and quadrupolar coupling constants ($\chi_Q$) for arbitrary spin $I > 1/2$ nuclei.

---

## Theoretical Background

### Model Hamiltonian

The nuclear quadrupole interaction Hamiltonian ($\hat{\mathcal{H}}_Q$) describes the coupling between a non-spherical nucleus
(with spin $I > 1/2$ and electric quadrupole moment $Q$) and the local electric field gradient (EFG) generated by its surrounding
electronic environment:

$$\hat{\mathcal{H}}_Q = \sum_{i}^{N_{\mathrm{nuc}}}\frac{eQ^i(1-\gamma_\infty^i)}{\hbar\,2I(2I-1)} \sum_{\alpha\beta} V_{\alpha\beta}^{i} \hat{I}_\alpha^i \hat{I}_\beta^i, \quad \alpha, \beta = x, y, z$$

where for $i$-th quadrupolar nuclear site:

- $Q$ is the nuclear electric quadrupole moment
- $V_{\alpha \beta}$ is the EFG tensor
- $\hat{I}_{\alpha, \beta}$ are the nuclear spin operators.
- $\gamma_\infty$ is the Sternheimer antishielding factor.
- $e$ is elementary charge.

Here $V_{\alpha\beta} \equiv \partial_{\alpha}\partial_{\beta} V(\mathbf{r})$, with $V(\mathbf{r})$ being the electrostatic potential evaluated at the nucleus. Notice that since the electric field is $\mathbf{E} = -\nabla V(\mathbf{r})$, the EFG tensor can also be expressed as $V_{\alpha\beta} = -\partial_{\alpha}E_{\beta}$.

To obtain the EFG, we start from the electrostatic potential $V$ centered at the nuclear site $\mathbf{r}_0$,  $V(\mathbf{r}_0)$;

$$V(\mathbf{r}_0)=\frac{1}{4\pi\varepsilon_0} \int d\mathbf{r}' \frac{\rho(\mathbf{r}')}{\lvert\mathbf{r}_0-\mathbf{r}'\rvert} \quad,$$


where $\rho(\mathbf{r})$ is the nuclear charge density.


### Point-Charge EFG Model

In an ionic crystal, the EFG at a particular site depends on the charge distribution of the surrounding ions. The simplest model treats the ions as stationary point charges located at lattice sites.

Assuming a collection of stationary point charges $\rho(\mathbf{r}') = \sum_k q_i \delta(\mathbf{r}' - \mathbf{r}_i)$, substituting this into the electrostatic potential integral yields:

$$V(\mathbf{r}_0) = \frac{1}{4\pi\varepsilon_0} \int d\mathbf{r}' \frac{\sum_k q_i \delta(\mathbf{r}' - \mathbf{r}_i)}{\lvert\mathbf{r}_0 - \mathbf{r}'\rvert} = \frac{1}{4\pi\varepsilon_0} \sum_i \frac{q_i}{\lvert\mathbf{r}_0 - \mathbf{r}_i\rvert} = \frac{1}{4\pi\varepsilon_0} \sum_i \frac{q_i}{\lvert \mathbf{x}_i \rvert}$$


where $q_{i}$ and $\mathbf{x}_i = \mathbf{r}_0 - \mathbf{r}_i$ are the charge and displacement vector of the site $i$-th located at distance $r_i = \lvert \mathbf{x}_i \rvert$ from the the site of interest ($\mathbf{r}_0$).


The EFG tensor components $V_{\alpha\beta} = \partial^2 V / \partial x_\alpha \partial x_\beta$ at the site of interest due to this periodic array of point charges are given by:

$$V_{\alpha\beta} = \frac{1}{4\pi\varepsilon_0} \sum_i q_i \left( \frac{3x_{i\alpha}x_{i\beta}-r_i^2\delta_{\alpha\beta}}{r_i^5} \right), \quad \alpha, \beta = x, y, z,$$


$$r_{i} = \lvert \mathbf{x}_{i} \rvert \equiv \sqrt{x_{i1}^2 + x_{i2}^2 + x_{i3}^2} \equiv \sqrt{x_{i}^2 + y_{i}^2 + z_{i}^2}.$$


where $\delta_{\alpha\beta}$ is the Kronecker delta, $r_i = \lvert \mathbf{x}_i \rvert$ and the sum runs over all charge sites within a sphere of chosen radius.



### Sternheimer Antishielding Correction

To account for the polarization of the core electronic cloud surrounding the probe nucleus, the lattice EFG is scaled using the Sternheimer antishielding factor $\gamma_\infty$:

$$V_{\alpha\beta}^{\mathrm{total}} = (1-\gamma_\infty)\, V_{\alpha\beta}^{\mathrm{lattice}}$$


### Calculated Properties

Diagonalization of the EFG tensor yields its principal components $(V_{xx}, V_{yy}, V_{zz})$, ordered by absolute magnitude:

$$\lvert V_{zz} \rvert \ge \lvert V_{yy} \rvert \ge \lvert V_{xx} \rvert$$

From these components, the quadrupolar parameters are derived:

$$\eta = \frac{V_{xx} - V_{yy}}{V_{zz}}, \qquad \chi_Q = \frac{e Q V_{zz}}{h}$$

$$\nu_z = \frac{3 e Q V_{zz}}{2I(2I - 1)h}, \qquad \nu_Q = \left\lvert \nu_z \sqrt{1 + \frac{\eta^2}{3}} \right\rvert$$


$$\qquad \nu_x = \frac{1}{2}\nu_z(\eta - 1), \qquad \nu_y = -\frac{1}{2}\nu_z(\eta + 1)$$

---


### Derivation of the Point-Charge EFG Tensor

**1. Electrostatic Potential**

The electrostatic potential at a probe site $\mathbf{r}_0$ due to a collection of point charges $q_k$ located at $\mathbf{r}_k$ is:

$$V(\mathbf{r}_0) = \frac{1}{4\pi\varepsilon_0} \sum_k \frac{q_k}{r_k}$$

where $r_k = \lvert \mathbf{x}_k \rvert$ and the displacement vector pointing from the probe site to the charge is $\mathbf{x}_k = \mathbf{r}_k - \mathbf{r}_0 \equiv (x_{1k}, x_{2k}, x_{3k})$.

---

**2. First Derivative (Electric Field)**

Taking the first spatial derivative of $V$ with respect to the coordinate $x_i$:

$$\frac{\partial V}{\partial x_i} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \, \frac{\partial}{\partial x_i}\left( r_k^{-1} \right)$$

Using the chain rule where $\frac{\partial r_k}{\partial x_i} = -\frac{x_{ik}}{r_k}$:

$$\frac{\partial V}{\partial x_i} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \, \frac{x_{ik}}{r_k^3}$$

---

**3. Second Derivative (EFG Tensor Components)**

The Electric Field Gradient tensor components $V_{ij}$ represent the second partial derivatives of the potential:

$$V_{ij} = \frac{\partial^2 V}{\partial x_i \partial x_j} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \, \frac{\partial}{\partial x_j} \left( \frac{x_{ik}}{r_k^3} \right)$$

Applying the product rule to $\frac{x_{ik}}{r_k^3} = x_{ik} \cdot r_k^{-3}$:

$$\frac{\partial}{\partial x_j} \left( x_{ik} \cdot r_k^{-3} \right) = \left( \frac{\partial x_{ik}}{\partial x_j} \right) r_k^{-3} + x_{ik} \left( \frac{\partial (r_k^{-3})}{\partial x_j} \right)$$

Substituting the coordinate derivative ($\frac{\partial x_{ik}}{\partial x_j} = \delta_{ij}$) and chain rule expansion ($\frac{\partial (r_k^{-3})}{\partial x_j} = \frac{3x_{jk}}{r_k^5}$):

$$\frac{\partial}{\partial x_j} \left( \frac{x_{ik}}{r_k^3} \right) = \frac{\delta_{ij}}{r_k^3} - \frac{3x_{ik}x_{jk}}{r_k^5}$$

Putting everything over a common denominator $r_k^5$ yields the final expression:

$$V_{ij} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \left( \frac{3x_{ik}x_{jk} - \delta_{ij}r_k^2}{r_k^5} \right), \quad i, j = 1, 2, 3$$


<!-- 

Assuming a collection of point charges $\rho(\mathbf{r})=\sum_k q_k\delta(\mathbf{r}-\mathbf{r}_k)$, the potential simplifies to:

$$V(\mathbf{r}_0)= \frac{1}{4\pi\varepsilon_0} \sum_k \frac{q_k}{\lvert \mathbf{x}_k \rvert}$$

$$\implies \quad \delta(\mathbf{r}'-\mathbf{r}_k) \ne 0 \quad \text{iff} \quad \mathbf{r}'=\mathbf{r}_k$$


The electrostatic potential at a probe position $\mathbf{r}_0$ is:

$$V(\mathbf{r}_0)=\frac{1}{4\pi\varepsilon_0} \int \frac{\rho(\mathbf{r}')}{\vert{}\mathbf{r}'-\mathbf{r}_0\vert{}}\,d\tau'$$

Assuming a collection of point charges $\rho(\mathbf{r})=\sum_k q_k\delta(\mathbf{r}-\mathbf{r}_k)$, the potential simplifies to:

$$V(\mathbf{r}_0)= \frac{1}{4\pi\varepsilon_0} \sum_k \frac{q_k}{R_k}$$

where $\mathbf{R}_k = \mathbf{r}_0 - \mathbf{r}_k$ and $R_k = \vert{}\mathbf{R}_k\vert{}$.

The EFG tensor is defined as the Hessian of the electrostatic potential:

$$V_{ij} = \frac{\partial^2 V}{\partial x_i\partial x_j}$$

Evaluating the partial derivatives yields the explicit sum over point charges:

$$V_{ij} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \left( \frac{3R_{k,i}R_{k,j}-\delta_{ij}R_k^2}{R_k^5} \right)$$

where $\delta_{ij}$ is the Kronecker delta.

### Sternheimer Antishielding Correction

To account for the polarization of the core electronic cloud surrounding the probe nucleus, the lattice EFG is scaled using the Sternheimer antishielding factor $\gamma_\infty$:

$$V_{ij}^{\mathrm{total}} = (1-\gamma_\infty)\, V_{ij}^{\mathrm{lattice}}$$

### Quadrupolar Interaction

For nuclei with spin $I > \frac{1}{2}$, the electric quadrupole interaction contribution to the Hamiltonian is:

$$\hat{\mathcal{H}}_Q = \sum_{i}^{N_{\mathrm{nuc}}}\frac{eQ^i(1-\gamma_\infty^i)}{\hbar\,2I(2I-1)} \sum_{\alpha\beta} V_{\alpha\beta}^{i} \hat{I}_\alpha^i \hat{I}_\beta^i$$

where:
- $Q^i$ is the $i$-th nuclear electric quadrupole moment.
- $V_{\alpha\beta}^{i}$ is the external EFG tensor at the site of the $i$-th quadrupolar nucleus.
- $\hat{I}_\alpha^i$ are the nuclear spin operators.

Diagonalization of the EFG tensor yields its principal components $(V_{xx}, V_{yy}, V_{zz})$, ordered by magnitude:

$$\vert{}V_{zz}\vert{} \ge \vert{}V_{yy}\vert{} \ge \vert{}V_{xx}\vert{}$$

From these components, the asymmetry parameter $\eta$ and quadrupolar coupling constant $\chi_Q$ are calculated:

$$\eta = \frac{V_{xx} - V_{yy}}{V_{zz}}, \qquad \chi_Q = \frac{e Q V_{zz}}{h}$$

---
-->